In [0]:
%sql
SELECT * FROM retail_analytics.reatail_dev.silver_retail_clean

In [0]:
%sql

CREATE OR REPLACE TABLE retail_analytics.reatail_dev.dim_product
(StockCode String PRIMARY KEY, Description String) 
USING DELTA

In [0]:
%sql
INSERT INTO retail_analytics.reatail_dev.dim_product
SELECT DISTINCT StockCode, Description
FROM retail_analytics.reatail_dev.silver_retail_clean
    

In [0]:
%sql

SELECT * FROM retail_analytics.reatail_dev.dim_product

In [0]:
%sql

CREATE OR REPLACE TABLE retail_analytics.reatail_dev.dim_customers 
(Customer_id string PRIMARY KEY)
USING DELTA;
    
INSERT INTO retail_analytics.reatail_dev.dim_customers
SELECT Distinct REPLACE(Customer_id,'.0', '') as Customer_id
FROM retail_analytics.reatail_dev.silver_retail_clean
WHERE Customer_id is not NULL

In [0]:
%sql
SELECT * FROM retail_analytics.reatail_dev.dim_customers

In [0]:
%sql
CREATE OR REPLACE TABLE retail_analytics.reatail_dev.dim_date
USING DELTA
AS
SELECT DISTINCT
    CAST(InvoiceDate AS DATE) AS date,
    YEAR(InvoiceDate) AS year,
    MONTH(InvoiceDate) AS month,
    DATE_FORMAT(InvoiceDate, 'MMMM') AS month_name,
    QUARTER(InvoiceDate) AS quarter
FROM retail_analytics.reatail_dev.silver_retail_clean
WHERE InvoiceDate IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE retail_analytics.reatail_dev.fact_sales
USING DELTA
AS 
SELECT Invoice as Invoice_number, 
       StockCode,
       REPLACE(Customer_id,'.0', '') as Customer_id,
       CAST(InvoiceDate AS DATE) as Invoice_date,
       Quantity as quantity,
       Price as price,
       ROUND(Quantity * Price,3) AS sales_value,
       Country as country,
       record_type
FROM retail_analytics.reatail_dev.silver_retail_clean

In [0]:
%sql
SELECT * FROM retail_analytics.reatail_dev.fact_sales WHERE Invoice_number = '489488'
AND StockCode = '22125'


In [0]:
%sql
SELECT c.customer_id , s.* FROM retail_analytics.reatail_dev.dim_customers c
inner join retail_analytics.reatail_dev.fact_sales s ON c.Customer_id = s.Customer_id